# Track B Colab Runner (Streaming Evaluation)

Evaluates your fine-tuned Track B checkpoint on streamed WikiArt samples and writes merge-compatible outputs.

## 0) Runtime

- Runtime -> Change runtime type -> GPU
- High-RAM runtime recommended

In [ ]:
# 0.5) Create Track B workspace in /content and show upload checklist
from pathlib import Path
import os

WORK_DIR = Path('/content/trackb_workspace')
folders = [
    WORK_DIR,
    WORK_DIR / 'scripts',
    WORK_DIR / 'artifacts',
    WORK_DIR / 'artifacts' / 'runs',
    WORK_DIR / 'artifacts' / 'checkpoints',
    WORK_DIR / 'artifacts' / 'checkpoints' / 'trackB_laion_vitl14',
    WORK_DIR / 'data',
    WORK_DIR / 'data' / 'labeled',
    WORK_DIR / 'logs',
    WORK_DIR / 'notebooks',
]
for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

placeholder_files = [
    WORK_DIR / 'scripts' / 'benchmark_track_b_finetuned.py',
    WORK_DIR / 'scripts' / 'run_track_b_comparison.py',
    WORK_DIR / 'scripts' / 'merge_track_a_runs.py',
    WORK_DIR / 'requirements.txt',
    WORK_DIR / 'artifacts' / 'checkpoints' / 'trackB_laion_vitl14' / 'classifier_head.pt',
    WORK_DIR / 'artifacts' / 'checkpoints' / 'trackB_laion_vitl14' / 'training_config.json',
]
for file_path in placeholder_files:
    file_path.touch(exist_ok=True)

(WORK_DIR / 'artifacts' / 'checkpoints' / 'trackB_laion_vitl14' / 'clip_finetuned').mkdir(parents=True, exist_ok=True)

os.chdir(WORK_DIR)
print('Workspace ready:', WORK_DIR)
print('CWD:', Path.cwd())

print('\nUpload/replace these files in place:')
print('  scripts/benchmark_track_b_finetuned.py')
print('  scripts/run_track_b_comparison.py')
print('  scripts/merge_track_a_runs.py')
print('  requirements.txt')
print('  artifacts/checkpoints/trackB_laion_vitl14/classifier_head.pt')
print('  artifacts/checkpoints/trackB_laion_vitl14/training_config.json')
print('  artifacts/checkpoints/trackB_laion_vitl14/clip_finetuned/*')

print('\nRaw WikiArt parquet upload is NOT required (streaming mode is used).')

In [ ]:
# 1) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Install dependencies
%pip install -q -r requirements.txt sentencepiece protobuf

In [ ]:
# 3) Optional: set HF token for streaming reliability
import os
from getpass import getpass
tok = getpass('HF_TOKEN (optional): ').strip()
if tok:
    os.environ['HF_TOKEN'] = tok
    print('HF_TOKEN set for this runtime session')
else:
    print('No HF token set; continuing unauthenticated')

In [ ]:
# 4) Preflight checks
from pathlib import Path
required = [
    'scripts/benchmark_track_b_finetuned.py',
    'scripts/run_track_b_comparison.py',
    'scripts/merge_track_a_runs.py',
    'artifacts/checkpoints/trackB_laion_vitl14/classifier_head.pt',
    'artifacts/checkpoints/trackB_laion_vitl14/training_config.json',
    'artifacts/checkpoints/trackB_laion_vitl14/clip_finetuned/config.json',
]
missing = [p for p in required if not Path(p).exists()]
print('Missing required files:', missing if missing else 'None')
if missing:
    raise FileNotFoundError('Upload missing files before running Track B evaluation.')

In [ ]:
# 5) Configure run parameters
STREAMING_REPO = 'huggan/wikiart'
STREAMING_SPLIT = 'train'
CHECKPOINT_DIR = 'artifacts/checkpoints/trackB_laion_vitl14'
EVAL_MAX_IMAGES = 4752   # set 0 for full stream
RUN_ID = ''               # optional custom run id

print('STREAMING_REPO:', STREAMING_REPO)
print('CHECKPOINT_DIR:', CHECKPOINT_DIR)
print('EVAL_MAX_IMAGES:', EVAL_MAX_IMAGES)

In [ ]:
# 6) Smoke run
!python scripts/run_track_b_comparison.py \
  --streaming-repo-id {STREAMING_REPO} \
  --streaming-split {STREAMING_SPLIT} \
  --checkpoint-dir {CHECKPOINT_DIR} \
  --output-root artifacts/runs \
  --max-images 64 \
  --run-id trackB_smoke_colab

In [ ]:
# 7) Full Track B evaluation run
run_id_arg = f'--run-id {RUN_ID}' if RUN_ID.strip() else ''
cmd = f"python scripts/run_track_b_comparison.py --streaming-repo-id {STREAMING_REPO} --streaming-split {STREAMING_SPLIT} --checkpoint-dir {CHECKPOINT_DIR} --output-root artifacts/runs --max-images {EVAL_MAX_IMAGES} {run_id_arg}"
print(cmd)
!{cmd}

In [ ]:
# 8) Optional: merge Track A + B + C (if those runs exist in this runtime)
from pathlib import Path
have_a = any(Path('artifacts/runs').glob('trackA_*'))
have_c = any(Path('artifacts/runs').glob('trackC_*'))
print('Track A present:', have_a, '| Track C present:', have_c)

if have_a or have_c:
    get_ipython().system('python scripts/merge_track_a_runs.py --runs-root artifacts/runs --run-globs trackA_* trackB_* trackC_* --output-dir artifacts/runs/trackABC_merged')
else:
    print('Skipping merged table: no trackA_* or trackC_* folders found in this runtime.')

In [ ]:
# 9) Save outputs to Drive
from pathlib import Path
import shutil

DRIVE_OUT = Path('/content/drive/MyDrive/clip_categorizer_outputs')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

for run_dir in Path('artifacts/runs').glob('trackB_*'):
    dest = DRIVE_OUT / 'runs' / run_dir.name
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(run_dir, dest)

merge_src = Path('artifacts/runs/trackABC_merged')
if merge_src.exists():
    merge_dst = DRIVE_OUT / 'trackABC_merged'
    if merge_dst.exists():
        shutil.rmtree(merge_dst)
    shutil.copytree(merge_src, merge_dst)

print('Saved outputs to:', DRIVE_OUT)

In [ ]:
# 10) Zip + auto-download to laptop
from pathlib import Path
from google.colab import files

zip_path = Path('artifacts/trackB_outputs.zip')
if zip_path.exists():
    zip_path.unlink()

!cd artifacts && zip -r trackB_outputs.zip runs/trackB_* runs/trackABC_merged || true

if zip_path.exists():
    print('Downloading:', zip_path)
    files.download(str(zip_path))
else:
    print('Zip file was not created; check artifacts/runs/trackB_*')

In [ ]:
# 11) Show key output locations
from pathlib import Path
print('Track B runs:')
for p in sorted(Path('artifacts/runs').glob('trackB_*')):
    print(' -', p)

summary_files = sorted(Path('artifacts/runs').glob('trackB_*/summary/track_b_summary.json'))
print('\nTrack B summaries:')
for p in summary_files:
    print(' -', p)

merged = Path('artifacts/runs/trackABC_merged/final_slides_table.csv')
if merged.exists():
    print('\nA+B+C merged table:', merged)